In [ ]:
# %% [markdown]
# # 🚀 Final Capstone: End-to-End Customer Churn Prediction
# **Author:** Data Science Intern
# **Objective:** Predict customer churn using Machine Learning and deploy as a web application.

# %%
# 1. Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Import our custom module from the src/ folder
import sys
sys.path.append(os.path.abspath('src'))
from preprocess import clean_and_prepare_data

sns.set_theme(style="whitegrid")

# %%
# 2. Load & Clean Data (Using our src module)
print("Loading and preprocessing data...")
df = clean_and_prepare_data('data/customer_churn.csv')

# Define features for the model
features = ['Tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'Avg_Spend', 'SeniorCitizen']
X = df[features]
y = df['Churn']

print(f"Dataset Shape: {X.shape}")

# %%
# 3. Scale & Split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# %%
# 4. Train Professional Model
print("Training Random Forest Classifier...")
model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# %%
# 5. Model Evaluation
print(f"\n✅ Model Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Plot Confusion Matrix
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Retained', 'Churned'], yticklabels=['Retained', 'Churned'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# Plot Feature Importance
importances = model.feature_importances_
sns.barplot(x=importances, y=features, palette='viridis')
plt.title("Feature Importance in Predicting Churn")
plt.show()

# %%
# 6. Export for Deployment
os.makedirs('deployment', exist_ok=True)
with open('deployment/model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('deployment/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("\n💾 Model and Scaler exported to /deployment successfully!")